In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch
from torch.optim import SGD, Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import uniform
from skorch import NeuralNetRegressor
from transform import new_Xtrain, new_Xval, Y_train, Y_val, X_test, Y_test
from sklearn.decomposition import PCA

In [2]:
new_Xtrain

,cat__VehBrand_B1,cat__VehBrand_B10,cat__VehBrand_B11,cat__VehBrand_B12,cat__VehBrand_B13,cat__VehBrand_B14,cat__VehBrand_B2,cat__VehBrand_B3,cat__VehBrand_B4,cat__VehBrand_B5,...,num__IDpol,num__Exposure,num__VehPower,num__BonusMalus,num__VehAge_log,num__DrivAge_log,num__Density_log,PC1,PC2,PC3
90929,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2187446.0,1.00,6,106,2.564949,3.526361,3.258097,46.193698,-0.295825,-3.451562
421433,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1119798.0,0.23,5,50,2.708050,3.871201,4.927254,-9.760485,-1.635755,-0.831414
336868,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3022812.0,0.03,12,50,0.693147,3.931826,4.094345,-9.853251,5.290522,-2.137381
520183,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2104998.0,0.13,4,50,3.044522,3.988984,7.012115,-9.718291,-2.453456,1.305697
302102,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,3067263.0,1.00,7,50,2.484907,3.688879,4.477337,-9.786441,0.319647,-1.440419
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110268,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5068503.0,0.89,4,50,0.693147,3.891820,5.170484,-9.756061,-2.583412,-0.362660
259178,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4022842.0,0.04,9,50,0.000000,4.204693,5.308268,-9.810698,2.418444,-0.617499
365838,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6100568.0,0.19,8,50,1.386294,3.970292,2.197225,-9.839474,1.130927,-3.717020
131932,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,2282710.0,0.08,4,100,2.397895,3.970292,8.227910,40.285681,-1.901961,1.770978


In [3]:
#Preparing the data
num_features = ['Exposure', 'VehPower', 'BonusMalus', 'VehAge_log', 'DrivAge_log', 'Density_log']
scaler = StandardScaler()
scaler.fit(new_Xtrain)
ready_Xtrain = scaler.transform(new_Xtrain)
ready_Xval = scaler.transform(new_Xval)
ready_test_X = scaler.transform(X_test)


In [4]:
class ModelDataset(Dataset):
    def __init__(self,x,y):
        self.x = x
        self.y = y
    def __getitem__(self,idx):
        return self.x[idx], self.y[idx]
    
    def __len__(self):
        return len(self.x)

In [5]:
X_train = ready_Xtrain.astype("float32")
X_val = ready_Xval.astype("float32")
Y_train = Y_train.astype("float32")
Y_val = Y_val.astype("float32")

In [6]:
X_train_np = X_train#.to_numpy()
y_train_np = Y_train.to_numpy().reshape(-1, 1)

X_test_np = X_val#.to_numpy()
y_test_np = Y_val.to_numpy().reshape(-1, 1)

ds = ModelDataset(torch.from_numpy(X_train_np),torch.from_numpy(y_train_np))
ds_test = ModelDataset(torch.from_numpy(X_test_np),torch.from_numpy(y_test_np))

train_loader = DataLoader(ds, batch_size=516, shuffle=True)
test_loader = DataLoader(ds_test, batch_size=1, shuffle=True)

In [7]:
#This is our network

input_dim = X_train.shape[1]

class SimpleNN(nn.Module):
    def __init__(self, activation=nn.ReLU):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 36)
        self.fc2 = nn.Linear(36, 24)
        self.fc3 = nn.Linear(24,12)
        self.fc4 = nn.Linear(12, 1)
        self.activation = activation() 
    def forward(self, x):
        #x = x.view(-1, 64)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.activation(self.fc3(x))
        x = self.fc4(x)
        return x
    
model = SimpleNN()

In [8]:
#We perform gridsearch to find the best parameters of the model
param_distributions = {
    'lr': [0.0001, 0.001],
    'optimizer': [Adam],
    'max_epochs': [10, 20, 30, 50],
    'batch_size': [128, 516],
    'module__activation': [nn.ReLU],
    'optimizer__weight_decay': [0.0, 0.0001 ],
}
from skorch import NeuralNetRegressor 
net = NeuralNetRegressor(module=SimpleNN)

grid_search = GridSearchCV(
    net,
    param_distributions,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=2
)
grid_search.fit(X_train_np, y_train_np)

print(grid_search.best_params_)
best_net = grid_search.best_estimator_

Fitting 3 folds for each of 32 candidates, totalling 96 fits
  epoch    train_loss    valid_loss      dur
-------  ------------  ------------  -------
      1        0.0572        0.0530  13.3924
      2        0.0550        0.0522  12.5549
      3        0.0539        0.0509  13.3381
      4        0.0526        0.0497  12.6609
      5        0.0514        0.0486  12.4045
      6        0.0504        0.0479  10.8406
      7        0.0498        0.0475  11.5768
      8        0.0495        0.0474  13.2988
      9        0.0493        0.0473  10.6928
     10        0.0492        0.0472  12.3583
     11        0.0491        0.0472  10.7790
     12        0.0490        0.0471  10.7888
     13        0.0489        0.0471  11.0259
     14        0.0489        0.0470  10.3966
     15        0.0488        0.0470  11.3211
     16        0.0487        0.0470  11.2029
     17        0.0487        0.0470  10.7580
     18        0.0486        0.0469  10.7424
     19        0.0486        0.0469  10

In [16]:
#with the best network found we do the prediction on the validation set
y_pred = best_net.predict(ready_Xval.astype("float32"))
print("Validation MSE:", mean_squared_error(Y_val, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(Y_val, y_pred)))
print("Test MAE:", mean_absolute_error(Y_val, y_pred)) 
print("Validation R^2:", r2_score(Y_val, y_pred))

Validation MSE: 0.048274021595716476
Test RMSE: 0.21971349889280012
Test MAE: 0.08831293135881424
Validation R^2: 0.163554847240448


In [22]:
test_prediction = best_net.predict(ready_test_X.astype("float32"))
print("Test MSE:", mean_squared_error(Y_test, test_prediction))
print("Test RMSE:", np.sqrt(mean_squared_error(Y_test, test_prediction)))
print("Test MAE:", mean_absolute_error(Y_test, test_prediction)) 
print("Test R²:", r2_score(Y_test, test_prediction))

Test MSE: 0.0509212501347065
Test RMSE: 0.22565737332227037
Test MAE: 0.08792843669652939
Test R²: 0.15423625707626343


The following is not relevant anymore, it was the model without gridsearch

In [ ]:
#Here we train the network
# criterion = nn.MSELoss()
# optimizer = Adam(model.parameters(), lr=0.0005)

# for epoch in range(50):
#     running_loss = 0.0
#     for i, data in enumerate(train_loader, 0):
#         inputs, labels = data
#         optimizer.zero_grad()
#         outputs = model(inputs.float())
#         loss = criterion(outputs, labels.float())
#         loss.backward()
#         optimizer.step()
#         running_loss += loss.item()
#     print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader)}")

Epoch 1, Loss: 0.055919279294546945
Epoch 2, Loss: 0.05094787175241109
Epoch 3, Loss: 0.04938079760267037
Epoch 4, Loss: 0.04898137868549254
Epoch 5, Loss: 0.04870681842516634
Epoch 6, Loss: 0.04853568305610264
Epoch 7, Loss: 0.048409043582030055
Epoch 8, Loss: 0.0482598140649719
Epoch 9, Loss: 0.048130943421075094
Epoch 10, Loss: 0.04804499825190279
Epoch 11, Loss: 0.0479363669108657
Epoch 12, Loss: 0.047795833785011566
Epoch 13, Loss: 0.04768449340046657
Epoch 14, Loss: 0.047596132504564403
Epoch 15, Loss: 0.04752073532876234
Epoch 16, Loss: 0.04742199040320342
Epoch 17, Loss: 0.04734739425008525
Epoch 18, Loss: 0.047222283942343644
Epoch 19, Loss: 0.04718834373936075
Epoch 20, Loss: 0.04711062035232109
Epoch 21, Loss: 0.04707193789243982
Epoch 22, Loss: 0.046985960504600466
Epoch 23, Loss: 0.046916271797984875
Epoch 24, Loss: 0.04685389724483671
Epoch 25, Loss: 0.046869924502374156
Epoch 26, Loss: 0.0467892187812524
Epoch 27, Loss: 0.04666238720002767
Epoch 28, Loss: 0.0466314890221

In [ ]:
# model.eval()
# with torch.no_grad():
#     outputs = model(torch.from_numpy(X_test_np).float())
#     mse = nn.MSELoss()(outputs, torch.from_numpy(y_test_np).float())
#     rmse = torch.sqrt(mse)
# print("MSE:", mse.item())
# print("RMSE:", rmse.item())

MSE: 0.04774697497487068
RMSE: 0.21851082146167755
